# Desafío · Una fuente cambió sin avisar

La aplicación publicó una versión nueva. El lote contiene un tipo de evento nuevo (`refund`), una columna adicional (`app_version`) y al menos un evento duplicado. Incorporalo sin perder la historia ni duplicar identificadores.

In [0]:
dbutils.widgets.text("student_id", "maxi_fronciani", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
print(challenge_path)

/Volumes/workspace/bigdata_maxi_fronciani/landing/events_json_v2


## Consigna 1 — Contrato y diagnóstico

Leé el lote nuevo, compará su esquema con `bronze_events` y respondé:

1. ¿Qué campo apareció?
2. ¿Qué valor de `event_type` no existía?
3. ¿Cuántas filas e identificadores distintos contiene?

In [0]:
from pyspark.sql import functions as F

challenge = spark.read.json(challenge_path)
historico = spark.table(f"{config.namespace}.bronze_events")

challenge.printSchema()

# 1. Campo nuevo
print("Campos nuevos:", set(challenge.columns) - set(historico.columns))

# 2. event_type nuevo
tipos_hist = {r[0] for r in historico.select("event_type").distinct().collect()}
tipos_nuevos = {r[0] for r in challenge.select("event_type").distinct().collect()}
print("event_type nuevos:", tipos_nuevos - tipos_hist)

# 3. Filas e identificadores distintos
display(challenge.agg(
    F.count("*").alias("filas"),
    F.countDistinct("event_id").alias("ids_distintos"),
))

root
 |-- app_version: string (nullable = true)
 |-- context: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)

Campos nuevos: {'app_version'}
event_type nuevos: {'refund'}


filas,ids_distintos
10001,10000


## Consigna 2 — Integración controlada

Construí `bronze_events_v2` con estas propiedades:

- Contiene los eventos históricos y el lote nuevo.
- Conserva `app_version`, usando `NULL` para la historia anterior.
- Tiene un único registro por `event_id`.
- Agrega `_source`, `_source_file` y `_ingested_at`.
- Puede volver a ejecutarse sin cambiar el total final.

No modifiques `bronze_events`: queremos conservarlo como evidencia de la primera ingesta.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

source_table = f"{config.catalog}.{config.schema}.bronze_events"
target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"

# Historia: ya tiene _source, _source_file e _ingested_at desde el notebook 01
historico = spark.table(source_table).withColumn("_prioridad", F.lit(1))

# Lote nuevo: le agregamos los mismos metadatos
nuevo = (spark.read.json(challenge_path)
    .withColumn("_source", F.lit("events_json_v2"))
    .withColumn("_source_file", F.lit(challenge_path))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_prioridad", F.lit(2)))

# Unir por nombre: la historia queda con app_version = NULL
unido = historico.unionByName(nuevo, allowMissingColumns=True)

# Un único registro por event_id (si se repite, gana el histórico)
w = Window.partitionBy("event_id").orderBy("_prioridad")
v2 = (unido
    .withColumn("_rn", F.row_number().over(w))
    .where("_rn = 1")
    .drop("_rn", "_prioridad"))

# overwrite: se puede re-ejecutar sin cambiar el total
(v2.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(target_table))

print(target_table, spark.table(target_table).count())

workspace.bigdata_maxi_fronciani.bronze_events_v2 210000


## Consigna 3 — Verificación automática

Completá las aserciones. Una entrega válida debe demostrar que no hay duplicados y que la columna evolucionada existe.

In [0]:
# Verificación automática de la integración
result = spark.table(target_table)
total_rows = result.count()
distinct_ids = result.select("event_id").distinct().count()
assert total_rows == distinct_ids, "Hay event_id duplicados"
assert "app_version" in result.columns, "No se preservó la evolución del esquema"
assert result.where("event_type = 'refund'").count() > 0, "No se incorporó refund"

## Reflexión final

En no más de 150 palabras, explicá la diferencia entre: detectar una evolución de esquema, aceptarla técnicamente y decidir que es válida para el negocio.